In [1]:
from fastafood import *

dna = DNASequence("ATGGCC")
print(dna)

DNASequence('ATGGCC')


In [3]:
gen = VariantGenerator(dna)
gen

In [10]:
snps = list(gen.generate_snps(protected_positions={0}))

In [11]:
snps

[DNASequence('AGGGCC'),
 DNASequence('AAGGCC'),
 DNASequence('ACGGCC'),
 DNASequence('ATAGCC'),
 DNASequence('ATTGCC'),
 DNASequence('ATCGCC'),
 DNASequence('ATGACC'),
 DNASequence('ATGTCC'),
 DNASequence('ATGCCC'),
 DNASequence('ATGGGC'),
 DNASequence('ATGGAC'),
 DNASequence('ATGGTC'),
 DNASequence('ATGGCG'),
 DNASequence('ATGGCA'),
 DNASequence('ATGGCT')]

In [12]:
dels = list(gen.generate_deletions(sizes=(1,2)))
dels

[DNASequence('TGGCC'),
 DNASequence('AGGCC'),
 DNASequence('ATGCC'),
 DNASequence('ATGCC'),
 DNASequence('ATGGC'),
 DNASequence('ATGGC'),
 DNASequence('GGCC'),
 DNASequence('AGCC'),
 DNASequence('ATCC'),
 DNASequence('ATGC'),
 DNASequence('ATGG')]

In [13]:
dups = list(gen.generate_duplications(sizes=(1,), times=3))
dups

[DNASequence('AAATGGCC'),
 DNASequence('ATTTGGCC'),
 DNASequence('ATGGGGCC'),
 DNASequence('ATGGGGCC'),
 DNASequence('ATGGCCCC'),
 DNASequence('ATGGCCCC')]

In [14]:
print("Protein:", dna.translate())
print("RevComp:", dna.reverse_complement())

Protein: MA
RevComp: DNASequence('GGCCAT')


In [ ]:
to_fasta(snps, "variants.fasta")

In [2]:
read_fasta("variants.fasta")

[DNASequence('AGGGCC'),
 DNASequence('AAGGCC'),
 DNASequence('ACGGCC'),
 DNASequence('ATAGCC'),
 DNASequence('ATTGCC'),
 DNASequence('ATCGCC'),
 DNASequence('ATGACC'),
 DNASequence('ATGTCC'),
 DNASequence('ATGCCC'),
 DNASequence('ATGGGC'),
 DNASequence('ATGGAC'),
 DNASequence('ATGGTC'),
 DNASequence('ATGGCG'),
 DNASequence('ATGGCA'),
 DNASequence('ATGGCT')]

In [3]:
count_fasta("variants.fasta")

15

In [4]:
list_fasta_headers("variants.fasta")

['seq_0',
 'seq_1',
 'seq_2',
 'seq_3',
 'seq_4',
 'seq_5',
 'seq_6',
 'seq_7',
 'seq_8',
 'seq_9',
 'seq_10',
 'seq_11',
 'seq_12',
 'seq_13',
 'seq_14']

In [14]:
from Bio import SeqIO
from Bio import Align

# 1. Chargement des données
ref_record = SeqIO.read("sge_test.fasta", "fasta")
sequences = list(SeqIO.parse("sge_test_snps_protected.fasta", "fasta"))

# 2. Configuration de l'aligneur
# On utilise "global" pour forcer l'alignement sur toute la longueur
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -0.5
aligner.extend_gap_score = -0.1

print(f"Référence: {ref_record.id}\n" + "="*30)

# 3. Alignement et affichage
for seq_record in sequences:
    alignments = aligner.align(ref_record.seq, seq_record.seq)
    # On prend le meilleur alignement trouvé
    best_alignment = alignments[0]
    
    print(f"\nAlignement de : {seq_record.id}")
    # L'objet alignment s'affiche naturellement sous forme de texte
    print(best_alignment)

Référence: original_sequence

Alignement de : original_sequence_0
target            0 CAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCAT
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 CAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCAT

target           60 TTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCA
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 TTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCA

target          120 TGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAG
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           120 TGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAG

target          180 TGGAAGATGGATTAGGAAGTCCTAAGCCTGAAGAAATTAAG 221
                180 ||||||||||||||||||||||||||||||||||||||||| 221
query           180 TGGAAGATGGATTAGGAAGTCCTAAGCCTGAAGAAATTAAG 221


Al

In [15]:
from Bio import SeqIO
from Bio import Align



# 2. Configuration de l'aligneur
aligner = Align.PairwiseAligner()
aligner.mode = 'global' # On reste en global pour mapper sur toute la longueur
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -1

print(f"Alignement sur : {ref_record.id}")
print(f"{'REFERENCE'.ljust(20)} | {ref_record.seq}")
print("-" * (23 + len(ref_record.seq)))

# 3. Alignement et extraction manuelle des caractères
for seq_record in sequences:
    alignments = aligner.align(ref_record.seq, seq_record.seq)
    best = alignments[0]
    
    # On récupère les deux séquences alignées (avec les gaps '-' ajoutés)
    # best[0] est la référence modifiée par l'alignement
    # best[1] est votre séquence (query) modifiée par l'alignement
    aligned_query = best[1]
    
    # Si la query est plus courte et que l'aligneur n'a pas complété 
    # jusqu'au bout de la réf, on remplit avec des espaces ou des points
    full_line = str(aligned_query).ljust(len(ref_record.seq), '.')

    print(f"{seq_record.id[:20].ljust(20)} | {full_line}")

Alignement sur : original_sequence
REFERENCE            | CAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCATTTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCATGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAGTGGAAGATGGATTAGGAAGTCCTAAGCCTGAAGAAATTAAG
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
original_sequence_0  | CAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCATTTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCATGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAGTGGAAGATGGATTAGGAAGTCCTAAGCCTGAAGAAATTAAG
original_sequence_1  | TAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCATTTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCATGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAGTGGAAGATGGATTAGGAAGTCCTAAGC

In [16]:
from Bio import SeqIO
from Bio import Align

# 1. Chargement (assumé déjà fait)
# ref_record = SeqIO.read("reference.fasta", "fasta")
# sequences = list(SeqIO.parse("sequences.fasta", "fasta"))

# 2. Configuration de l'aligneur
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -1

# Préparation de la référence en string pour la comparaison
ref_seq = str(ref_record.seq)

print(f"Alignement (Différences uniquement) sur : {ref_record.id}")
print(f"{'REFERENCE'.ljust(20)} | {ref_seq}")
print("-" * (23 + len(ref_seq)))

# 3. Alignement et filtrage des bases identiques
for seq_record in sequences:
    alignments = aligner.align(ref_record.seq, seq_record.seq)
    best = alignments[0]
    
    # On récupère les deux brins alignés (ils ont maintenant la même longueur grâce aux gaps)
    aligned_ref = str(best[0])
    aligned_query = str(best[1])
    
    diff_line = []
    
    # On compare chaque base
    for r, q in zip(aligned_ref, aligned_query):
        if q == "-":           # C'est un gap dans la query (manquant)
            diff_line.append("-")
        elif r == q:           # C'est identique
            diff_line.append(".")
        else:                  # C'est une mutation (divergent)
            diff_line.append(q)
            
    # On convertit la liste en chaîne
    result_str = "".join(diff_line)
    
    # Optionnel : Si la séquence est plus courte que la référence originale,
    # on complète avec des points de vide (pour l'alignement global)
    result_str = result_str.ljust(len(ref_seq), '.')

    print(f"{seq_record.id[:20].ljust(20)} | {result_str}")

Alignement (Différences uniquement) sur : original_sequence
REFERENCE            | CAGTTGCAGACTTGCAAAGGATGTTTCCCACTCCACCATCTTTGGAACAGCATCCTGCATTTTCTCCTGTGATGAATTATAAAGATGGGATCAGCTCAGAGACAGTGACAGCATTAGGCATGATGGAGAGCCCTATGGTCAGTATGGTTTCAACACAACTCACAGAATTCAAAATGGAAGTGGAAGATGGATTAGGAAGTCCTAAGCCTGAAGAAATTAAG
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
original_sequence_0  | .............................................................................................................................................................................................................................
original_sequence_1  | T.....................................................................................................................................................................................

In [20]:
from Bio import SeqIO
from Bio import Align
from PIL import Image, ImageDraw, ImageFont

# 1. Configuration de l'aligneur (identique à ton code)
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -1

# 2. Préparation des données
ref_record = SeqIO.read("sge_test.fasta", "fasta")
sequences = list(SeqIO.parse("sge_test_dup1.fasta", "fasta"))
ref_seq = str(ref_record.seq)

rows = [(ref_record.id, ref_seq)] # Liste pour stocker (ID, Séquence traitée)

for seq_record in sequences:
    alignments = aligner.align(ref_record.seq, seq_record.seq)
    best = alignments[0]
    aligned_ref = str(best[0])
    aligned_query = str(best[1])
    
    diff_line = ""
    for r, q in zip(aligned_ref, aligned_query):
        if q == "-": diff_line += "-"
        elif r == q: diff_line += "."
        else: diff_line += q
    rows.append((seq_record.id, diff_line))

# 3. Génération de l'image
char_width = 10
char_height = 18
margin = 20
label_width = 200

img_width = label_width + (len(ref_seq) * char_width) + (margin * 2)
img_height = (len(rows) * char_height) + (margin * 2)

# Création du canevas (fond blanc)
img = Image.new('RGB', (img_width, img_height), color=(255, 255, 255))
d = ImageDraw.Draw(img)

# Chargement d'une police monospace (importante pour l'alignement)
try:
    # Sur Windows: 'consola.ttf', sur Linux: 'DejaVuSansMono.ttf'
    font = ImageFont.truetype("consola.ttf", 15)
except:
    font = ImageFont.load_default()

# Dessin de l'alignement
for i, (label, seq) in enumerate(rows):
    y_pos = margin + (i * char_height)
    
    # Dessiner l'ID de la séquence (en gras/noir)
    d.text((margin, y_pos), label[:20], fill=(0, 0, 0), font=font)
    
    # Dessiner la séquence
    for j, char in enumerate(seq):
        x_pos = margin + label_width + (j * char_width)
        
        # Coloration optionnelle : rouge pour les mutations, gris pour les points
        color = (200, 200, 200) if char == "." else (255, 0, 0)
        if i == 0: color = (0, 0, 255) # Référence en bleu
            
        d.text((x_pos, y_pos), char, fill=color, font=font)

# 4. Sauvegarde
img.save("alignement_output.png")
print("Image 'alignement_output.png' générée avec succès.")

Image 'alignement_output.png' générée avec succès.
